# 07 分群地圖

**用途**：將 K-means 與階層式分群結果畫到縣市地圖上。需要 geopandas。

| 項目 | 說明 |
|---|---|
| 輸入 | 分群結果（於本檔重算）、縣市界線 shapefile |
| 輸出 | 各 k 的分群地圖（存於 notebooks/_scratch/） |
| 來源 | MVA 期末報告分析.ipynb（Colab 原檔，191 cells） 之 cell 168–173（Update 05/29 最終版） |

> 原 cell 169（Colab 上載入 shapefile／GDAL）改由共用設定提供 taiwan_map 與 county_col。


In [ ]:
# 共用設定：載入資料並提供 df / pivot_table_age_std / pivot_table_cleaned / scaled_data / taiwan_map / prop 等變數
# （取代原 Colab 的 Drive 掛載、字型下載與 GDAL 設定；詳見 _legacy_setup.py）
import sys; sys.path.insert(0, '..')   # _legacy_setup.py 位於 notebooks/
from _legacy_setup import *

In [ ]:
# 前置狀態（原 Colab 於 05–06 產生）：06 輸出的階層式分群大表；若不存在，改用 src/04 產生的同等結果
import shutil
if not (SCRATCH / 'hierarchical_clustering_comprehensive_results.csv').exists():
    shutil.copy(ROOT / 'outputs/tables/hclust_cluster_assignments.csv', SCRATCH / 'hierarchical_clustering_comprehensive_results.csv')
scaled_clustering_data = scaled_data

### Map visualization

#### K-Means

In [ ]:
import os
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
from matplotlib.patches import Patch
from sklearn.cluster import KMeans

# 排除連江縣
taiwan_map_clean = taiwan_map[taiwan_map[county_col] != '連江縣'].copy()
taiwan_map_clean[county_col] = taiwan_map_clean[county_col].str.replace('臺', '台', 'one' if hasattr(str, 'replace_old') else -1)

# 確保沿用你先前的字型設定
if 'prop' in locals():
    plt.rcParams['font.family'] = prop.get_name()

# 確保縣市名稱清單 (對齊 K-means 順序)
counties_names = pivot_table_cleaned.index

# 固定群組顏色
color_dict = {
    'Group A': '#e41a1c',  # 紅
    'Group B': '#377eb8',  # 藍
    'Group C': '#4daf4a',  # 綠
    'Group D': '#984ea3',  # 紫
    'Group E': '#ff7f00'   # 橘
}

# ==========================================
# 2. 進入迴圈，分別繪製 k=2 與 k=5
# ==========================================
for k_val in [2, 5]:

    # 執行 K-Means 分群
    kmeans = KMeans(n_clusters=k_val, random_state=42, n_init=10)
    labels = kmeans.fit_predict(scaled_data)

    # 動態生成群組標籤
    groups_list = [f'Group {chr(65 + i)}' for i in range(k_val)]
    cluster_map = {i: groups_list[i] for i in range(k_val)}
    encoded_labels = [cluster_map[l] for l in labels]

    # 建立分群對照 DataFrame
    df_clusters = pd.DataFrame({
        'County': counties_names,
        'Cluster': encoded_labels
    })

    # 合併地圖與分群數據
    map_data = taiwan_map_clean.merge(df_clusters, left_on=county_col, right_on='County', how='left')
    map_data['color'] = map_data['Cluster'].map(color_dict).fillna('#cbd5e1')

    # ==========================================
    # 3. 開始建構圖表與子母畫面
    # ==========================================
    fig, ax_main = plt.subplots(figsize=(10, 12))

    # 🗺️ (A) 繪製主圖
    map_data.plot(ax=ax_main, color=map_data['color'], edgecolor='black', linewidth=0.4)

    # ✨ 關鍵修改：將經度左界從 120.0 放寬到 119.0，為左側小地圖拉出完美腹地
    ax_main.set_xlim(119.0, 122.3)
    ax_main.set_ylim(21.8, 25.4)

    ax_main.set_title(f'Taiwan K-Means Map\n(k={k_val})', fontsize=16, pad=20, fontweight='bold')
    ax_main.set_xlabel('Longitude (經度)', fontsize=11)
    ax_main.set_ylabel('Latitude (緯度)', fontsize=11)
    ax_main.grid(True, linestyle='--', alpha=0.3)

    for idx, row in map_data[map_data[county_col] != '金門縣'].iterrows():
        if pd.notnull(row['Cluster']):
            centroid = row['geometry'].centroid
            ax_main.annotate(text=row['County'], xy=(centroid.x, centroid.y), ha='center', fontsize=12,
                             bbox=dict(boxstyle='round,pad=0.1', fc='white', alpha=0.6, ec='none'))

    # 🗺️ (B) 嵌入金門小地圖
    # ✨ 改用 ax_main.inset_axes，[左邊距, 下邊距, 寬度, 高度] 比例完全鎖在主圖內
    ax_kinmen = ax_main.inset_axes([0.05, 0.72, 0.22, 0.22])
    kinmen_data = map_data[map_data[county_col] == '金門縣']
    kinmen_data.plot(ax=ax_kinmen, color=kinmen_data['color'], edgecolor='black', linewidth=0.5)
    ax_kinmen.set_xlim(118.1, 118.6)
    ax_kinmen.set_ylim(24.2, 24.6)
    ax_kinmen.set_title('金門縣', fontsize=12, fontweight='bold', color='#1e293b')
    ax_kinmen.set_xticks([])
    ax_kinmen.set_yticks([])

    # ==========================================
    # 4. 手動客製化圖例 (Legend)
    # ==========================================
    legend_elements = [
        Patch(facecolor=color_dict[group], edgecolor='black', linewidth=0.5, label=group)
        for group in groups_list
    ]
    ax_main.legend(handles=legend_elements, title='Clusters', loc='lower right', fontsize=12, title_fontsize=13)

    # 💾 儲存高解析度地圖
    plt.savefig(f'map_kmeans_k{k_val}.png', dpi=300, bbox_inches='tight')
    plt.show()

print("\n🎉 地圖（k=2, k=5）已繪製完成")

#### H-Clustering

In [ ]:
# 讀取上一階段打包好的 16 種組合分群結果 (以縣市名稱為 Index)
df_hierarchical = pd.read_csv('hierarchical_clustering_comprehensive_results.csv', index_col=0)

# 確保沿用你的中文字型設定
if 'prop' in locals():
    plt.rcParams['font.family'] = prop.get_name()

# 固定群組高對比顏色面版
color_dict = {
    'Group A': '#e41a1c',  # 紅
    'Group B': '#377eb8',  # 藍
    'Group C': '#4daf4a',  # 綠
    'Group D': '#984ea3',  # 紫
    'Group E': '#ff7f00'   # 橘
}

# ==========================================
# 2. 定義方案 A 與方案 B 的地圖任務清單
# 欄位命名規則必須完美對齊 CSV 欄位（例如：'Single_k2', 'Ward_k5'）
# ==========================================
map_tasks = [
    # 方案 A：所有方法皆看 k = 2
    {'scenario': 'A', 'method': 'Single',   'k': 2},
    {'scenario': 'A', 'method': 'Complete', 'k': 2},
    {'scenario': 'A', 'method': 'Average',  'k': 2},
    {'scenario': 'A', 'method': 'Ward',     'k': 2},

    # 方案 B：指定最佳/次佳群數 (前三者 k=4, Ward k=5)
    {'scenario': 'B', 'method': 'Single',   'k': 4},
    {'scenario': 'B', 'method': 'Complete', 'k': 4},
    {'scenario': 'B', 'method': 'Average',  'k': 4},
    {'scenario': 'B', 'method': 'Ward',     'k': 5},
]

# ==========================================
# 3. 進入自動化自動繪圖迴圈 (共 8 張圖)
# ==========================================
for task in map_tasks:
    scenario = task['scenario']
    method = task['method']
    k_val = task['k']

    # 對齊 CSV 的欄位名稱
    csv_col_name = f"{method}_k{k_val}"

    # 建立暫時的對照 DataFrame 準備進行 Merge
    df_temp = pd.DataFrame({
        'County': df_hierarchical.index,
        'Cluster': df_hierarchical[csv_col_name]
    })

    # 地圖與數據結合
    map_data = taiwan_map_clean.merge(df_temp, left_on=county_col, right_on='County', how='left')
    map_data['color'] = map_data['Cluster'].map(color_dict).fillna('#cbd5e1')

    # 建立畫布
    fig, ax_main = plt.subplots(figsize=(10, 12))

    # 🗺️ (A) 繪製主圖 (範圍包含台灣本島與澎湖縣)
    map_data.plot(ax=ax_main, color=map_data['color'], edgecolor='black', linewidth=0.3)
    ax_main.set_xlim(119.0, 122.3)  # 完美容納澎湖的寬闊經度
    ax_main.set_ylim(21.8, 25.4)

    # 動態精美標題
    ax_main.set_title(f'Taiwan Hierarchical Clustering Map\n{method} Linkage (k={k_val})',
                      fontsize=15, pad=15, fontweight='bold')
    ax_main.set_xlabel('Longitude (經度)', fontsize=11)
    ax_main.set_ylabel('Latitude (緯度)', fontsize=11)
    ax_main.grid(True, linestyle='--', alpha=0.3)

    for idx, row in map_data[map_data[county_col] != '金門縣'].iterrows():
        if pd.notnull(row['Cluster']):
            centroid = row['geometry'].centroid
            ax_main.annotate(text=row['County'], xy=(centroid.x, centroid.y), ha='center', fontsize=12,
                             bbox=dict(boxstyle='round,pad=0.1', fc='white', alpha=0.6, ec='none'))

    # 🗺️ (B) 僅保留金門小地圖 (利用 inset_axes 舒適安放於左上角海域)
    ax_kinmen = ax_main.inset_axes([0.05, 0.72, 0.22, 0.22])
    kinmen_data = map_data[map_data[county_col] == '金門縣']
    kinmen_data.plot(ax=ax_kinmen, color=kinmen_data['color'], edgecolor='black', linewidth=0.4)
    ax_kinmen.set_xlim(118.1, 118.6)
    ax_kinmen.set_ylim(24.2, 24.6)
    ax_kinmen.set_title('金門縣', fontsize=12, fontweight='bold', color='#1e293b')
    ax_kinmen.set_xticks([])
    ax_kinmen.set_yticks([])

    # ==========================================
    # 4. 手動客製化圖例 (僅顯示該 k 值範圍內的 Group)
    # ==========================================
    groups_list = [f'Group {chr(65 + i)}' for i in range(k_val)]
    legend_elements = [
        Patch(facecolor=color_dict[group], edgecolor='black', linewidth=0.5, label=group)
        for group in groups_list
    ]
    ax_main.legend(handles=legend_elements, title=f'Clusters (k={k_val})', loc='lower right', fontsize=12, title_fontsize=13)

    # 💾 5. 自動儲存高解析度圖片，檔名格式：map_hierarchical_ScenarioA_Single_k2.png
    file_name = f'map_hierarchical_{method}_k{k_val}.png'
    plt.savefig(file_name, dpi=300, bbox_inches='tight')
    plt.show()

    print(f"📸 已成功儲存地圖：{file_name}")

print("\n✨ 地圖已全部繪製並儲存完畢。")